# Семинар 4.
## R2D2

In [ ]:
import warnings

import arviz as az
import matplotlib.pyplot as plt
import numpy as np
# import nutpie # почему-то не подгружается, мб из-за рабочего WIFI
import pymc as pm
import scipy.special as sp
import scipy.stats as st
from sklearn.datasets import fetch_california_housing
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore")

In [ ]:
housing = fetch_california_housing(as_frame=True)

In [ ]:
data = housing["data"].copy()
target = housing["target"]
non_negative_variables = [
    "MedInc",
    "HouseAge",
    "AveRooms",
    "AveBedrms",
    "Population",
    "AveOccup",
]
data[non_negative_variables] = np.log1p(data[non_negative_variables])
target = np.log(target)

In [ ]:
Xscaler = StandardScaler()
Yscaler = StandardScaler()
X = Xscaler.fit_transform(data.values)
Y = Yscaler.fit_transform(target.values[:, None]).ravel()

### R2

Предполагаем, что на наших данных должнен быть в диапазоне 0.6-0.8

## Важность фичей (?)

In [ ]:
X.shape, Y.shape

In [ ]:
data.columns.size

In [ ]:
p = X.shape[1]

with pm.Model(
    coords=dict(
        variable=list(data.columns), 
        observation=np.arange(X.shape[0])
    )
) as model:
    # X_data = pm.Data("X", X, dims=("observation", "variable"))
     
    intercept = pm.Normal("intercept", 0, 0.5)
    σ = pm.LogNormal("σ", 0, 0.5)
    

    R2 = pm.Beta("R2", mu=0.6, sigma=0.2)
    ϕ = pm.Dirichlet("ϕ", np.ones(p), dims="variable")
    
    model_σ = σ * R2 ** .5
    resid_σ = σ * (1 - R2) ** .5
    
    β = pm.Normal("β", 0, model_σ * ϕ ** .5, dims="variable")
    
    μ = intercept + X @ β
    
    pm.Normal("obs", μ, resid_σ, dims="observation", observed=Y)

In [ ]:
pm.model_to_graphviz(model)

In [ ]:
with model:
    trace = pm.sample()

In [ ]:
az.plot_forest(trace, var_names=["ϕ"]);

In [ ]:
az.plot_forest(trace, var_names=["β"]);

In [ ]:
az.plot_trace_dist(trace, var_names=["β"], compact=False);
plt.tight_layout()

In [ ]:
data.columns[:-2]

In [ ]:
float(X[:, -2:].min()), float(X[:, -2:].max())

In [ ]:
M = int(1.75 * (4 * 1.2) / 0.1)
M